# Title

**Python-Based Stock Market Analysis of Major Technology Companies**

This notebook presents a simple financial data analysis project for ACC102 Track 2. It compares five large technology companies using five years of daily historical stock prices downloaded from Yahoo Finance.

## 1. Problem Definition

The purpose of this project is to examine how five major technology stocks have behaved over the last five years in terms of price trend, return, volatility, correlation, and drawdown. The main question is whether simple Python-based analysis can help a beginner understand differences in stock performance and risk more clearly.

## 2. Target User / Audience

This notebook is written for beginner retail investors and business students who want a practical example of data analysis applied to stock market data. For that reason, the methods are kept simple, visual, and easy to interpret.

## 3. Data Source

The dataset comes from **Yahoo Finance** through the `yfinance` Python library. The notebook downloads the most recent five years of daily historical data for these tickers:

- AAPL
- MSFT
- NVDA
- GOOGL
- AMZN

The analysis uses adjusted closing prices where available, because they are more suitable for historical comparison than raw closing prices alone.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from analysis import TICKERS, run_analysis

plt.style.use("default")
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


## 4. Data Loading and Preparation

The next cell runs the main workflow from `analysis.py`. This keeps the notebook consistent with the script version of the project and makes the repository easier to maintain. The workflow downloads the data, extracts the relevant price series, cleans missing values, creates output files, and prepares the tables used below.

In [ ]:
results = run_analysis()

prices = results["prices"]
daily_returns = results["daily_returns"]
cumulative_returns = results["cumulative_returns"]
rolling_volatility = results["rolling_volatility"]
ma20 = results["ma20"]
ma50 = results["ma50"]
correlation_matrix = results["correlation_matrix"]
drawdown = results["drawdown"]
summary_table = results["summary_table"]

print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
print(f"Observations: {len(prices)} trading days")
print(f"Tickers: {', '.join(TICKERS)}")

prices.head()


The table above shows the cleaned price data that will be used throughout the notebook. Each column represents one stock, and each row is a trading day.

## 5. Exploratory Analysis

This section compares overall stock price movement and cumulative return. Looking at both helps separate the absolute price level from actual investment growth over time.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in prices.columns:
    ax.plot(prices.index, prices[ticker], linewidth=1.8, label=ticker)
ax.set_title("Daily Price Trend for Five Technology Stocks", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Price ($)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


**Interpretation:** The chart shows that the five stocks do not move in exactly the same way over time. Even within one sector, price paths can differ substantially, which is one reason investors should compare multiple risk and return measures instead of looking only at headline company names.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in cumulative_returns.columns:
    ax.plot(cumulative_returns.index, cumulative_returns[ticker] * 100, linewidth=1.8, label=ticker)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Cumulative Return Comparison", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Cumulative Return (%)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


**Interpretation:** Cumulative return is more useful than price alone when comparing investment performance. A stock can have a high share price but still produce weaker growth than another stock with a lower share price.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for index, ticker in enumerate(daily_returns.columns):
    axes[index].hist(daily_returns[ticker] * 100, bins=40, color="steelblue", edgecolor="white", alpha=0.85)
    axes[index].set_title(ticker, fontweight="bold")
    axes[index].set_xlabel("Daily Return (%)")
    axes[index].set_ylabel("Frequency")
    axes[index].grid(alpha=0.3, axis="y")
axes[-1].set_visible(False)
fig.suptitle("Distribution of Daily Returns", fontweight="bold")
fig.tight_layout()
plt.show()


**Interpretation:** The distributions show that daily stock returns are usually clustered around zero, but larger positive and negative moves still occur. Wider distributions suggest greater day-to-day uncertainty.

## 6. Risk and Return Analysis

This section focuses more directly on uncertainty and downside risk. Rolling volatility helps show how unstable returns are over time, while moving averages provide a simple trend indicator. Correlation and drawdown help compare the stocks as a group.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for ticker in rolling_volatility.columns:
    ax.plot(rolling_volatility.index, rolling_volatility[ticker] * 100, linewidth=1.8, label=ticker)
ax.set_title("30-Day Rolling Volatility", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Annualized Volatility (%)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


**Interpretation:** Rolling volatility changes over time rather than staying constant. Periods with higher volatility represent greater short-term uncertainty, which can matter a lot for investors who are sensitive to sharp price swings.

In [ ]:
representative_ticker = "AAPL"
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(prices.index, prices[representative_ticker], label=f"{representative_ticker} Price", linewidth=1.5)
ax.plot(ma20.index, ma20[representative_ticker], label="20-Day MA", linewidth=1.8)
ax.plot(ma50.index, ma50[representative_ticker], label="50-Day MA", linewidth=1.8)
ax.set_title(f"{representative_ticker} Price with 20-Day and 50-Day Moving Averages", fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Price ($)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


**Interpretation:** Moving averages smooth short-term noise and make broader trend direction easier to see. They do not predict the future, but they are useful for describing whether recent market momentum has been stronger or weaker than the longer-term trend.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
image = ax.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_yticks(range(len(correlation_matrix.index)))
ax.set_xticklabels(correlation_matrix.columns)
ax.set_yticklabels(correlation_matrix.index)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
for row in range(len(correlation_matrix.index)):
    for col in range(len(correlation_matrix.columns)):
        ax.text(col, row, f"{correlation_matrix.iloc[row, col]:.2f}", ha="center", va="center", color="black")
ax.set_title("Correlation Matrix of Daily Returns", fontweight="bold")
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.show()


**Interpretation:** Positive correlation values indicate that the stocks often move in the same direction. This matters because owning several technology stocks may still leave an investor exposed to similar market shocks.

In [ ]:
summary_table


**Interpretation:** The summary table combines return and risk measures into one comparison. It is useful for identifying whether stronger historical performance came with higher volatility or deeper drawdowns.

In [ ]:
max_drawdown_table = (drawdown.min() * 100).to_frame(name="Maximum Drawdown (%)").round(2)
max_drawdown_table


**Interpretation:** Maximum drawdown shows the largest percentage fall from a previous peak. This gives a clearer view of downside experience than average return alone.

## 7. Key Insights

Several broad insights can be taken from this analysis. First, the five companies can produce noticeably different long-run returns even though they are all major technology firms. Second, higher return often comes with greater volatility and potentially larger drawdowns. Third, the correlation matrix suggests that these stocks are not fully independent from one another, so a portfolio made only of large technology stocks may still be concentrated in one part of the market. These points are useful for education, but they should not be treated as investment advice.

## 8. Limitations

This project has several limitations. It uses historical data only, so it cannot predict future returns. It includes only five companies from one sector, which limits broader generalisation. The analysis is based on price behaviour rather than company fundamentals, and the results depend on Yahoo Finance data being available at the time of download. In addition, simple statistics such as standard deviation and correlation do not fully capture all types of financial risk.

## 9. Conclusion

This notebook demonstrates a complete but manageable Python workflow for stock market analysis. By using common libraries and public data, it is possible to build a reproducible project that covers data loading, cleaning, transformation, visualisation, and interpretation. For an introductory audience, the combination of cumulative return, volatility, moving averages, correlation, and drawdown provides a practical starting point for understanding how large technology stocks behave over time.